# OpenAlex field vs Semantic Scholar field

How often the field we give an OpenAlex author (`scripts/openalex_cascade/fields.py`) agrees with the Semantic Scholar field, on authors that are almost surely the same person in both: a name that appears only once in each source and the same first-publication year (±2).

In [1]:
import sys
import pandas as pd

sys.path.insert(0, "../../..")
sys.path.insert(0, "../../../scripts/enrichment")
from openalex import normalize_name
from libs.utils.config import config_default

R = "../../../../results/summary"
FIELDS = ["Biology", "Computer Science", "Mathematics", "Physics", "Psychology", "Sociology"]

In [2]:
# Semantic Scholar: names with a single candidate researcher, with their field and first year
ss = pd.read_parquet(f"{R}/cascade/candidates_name.parquet", columns=["query", "Researcher_id", "Field"])
ss["norm"] = ss["query"].map(normalize_name)
ss = ss[ss.groupby("norm")["norm"].transform("size") == 1]
first_year = pd.read_parquet(config_default("ss_parquet"), columns=["Researcher_id", "First_year"]).drop_duplicates("Researcher_id")
ss = ss.merge(first_year, on="Researcher_id")
len(ss)

104193

In [3]:
# OpenAlex: names with a single candidate author
oa = pd.read_parquet(f"{R}/openalex_cascade/candidates.parquet", columns=["norm", "oa_id", "first_year"])
oa = oa[oa.groupby("norm")["norm"].transform("size") == 1]
len(oa)

104192

In [4]:
# Same person in both: unique name in each source and same first-publication year (±2)
pairs = ss[["norm", "Field", "First_year"]].merge(oa, on="norm")
pairs = pairs[(pairs["First_year"] - pairs["first_year"]).abs() <= 2]
len(pairs)

5671

In [5]:
# The two OpenAlex classifications of each author (written by 2_author_fields.py)
fields = pd.read_parquet(f"{R}/openalex_cascade/author_fields.parquet", columns=["oa_id", "field_topics", "field_concepts"])
pairs = pairs.merge(fields, on="oa_id", how="left")
pairs["both_agree"] = pairs["field_topics"].where(pairs["field_topics"] == pairs["field_concepts"])

rows = []
for rule in ["field_topics", "field_concepts", "both_agree"]:
    has = pairs[pairs[rule].notna()]
    rows.append({"rule": rule,
                 "authors with a field": f"{pairs[rule].notna().mean():.1%}",
                 "agreement with Semantic Scholar": f"{(has[rule] == has['Field']).mean():.1%}"})
pd.DataFrame(rows)

,rule,authors with a field,agreement with Semantic Scholar
0,field_topics,59.2%,72.4%
1,field_concepts,64.8%,77.8%
2,both_agree,40.8%,83.9%


In [6]:
# Agreement per Semantic Scholar field
pd.DataFrame({rule: pairs[pairs[rule].notna()].assign(ok=lambda d: d[rule] == d["Field"]).groupby("Field")["ok"].mean()
              for rule in ["field_topics", "field_concepts", "both_agree"]}).round(3)

,field_topics,field_concepts,both_agree
Field,,,
Biology,0.918,0.845,0.957
Computer Science,0.828,0.867,0.940
Mathematics,0.658,0.710,0.752
Physics,0.718,0.778,0.866
Psychology,0.415,0.727,0.705
Sociology,0.738,0.604,0.757


In [7]:
# Where they disagree (topics): rows = Semantic Scholar field, columns = OpenAlex field, % per row
has = pairs[pairs["field_topics"].notna()]
(pd.crosstab(has["Field"], has["field_topics"], normalize="index") * 100).round(1)

field_topics,Biology,Computer Science,Mathematics,Physics,Psychology,Sociology
Field,,,,,,
Biology,91.8,3.0,0.6,1.1,1.6,2.0
Computer Science,8.5,82.8,0.6,3.7,2.0,2.4
Mathematics,2.8,26.5,65.8,4.4,0.3,0.2
Physics,14.3,6.6,2.3,71.8,1.9,3.1
Psychology,31.6,7.2,0.9,0.4,41.5,18.4
Sociology,13.1,7.5,0.0,0.9,4.7,73.8
